# Música em Contexto: Music Brainz + Spotify

## Camada Bronze
Salvando as dataframes relevantes para o projeto. Depois acrescento os comentários para todas as tabelas. Ainda não é feita nenhuma transformação.

Importante: O JSON **artist** de Music Brainz continha campos com caracteres especiais que dificultavam a importação. Entre transformar esses campos para poder salvar todos os dados numa tabela e cortar os dados complexos desnecessários eu decidi pela segunda opção: Salvei uma tabela **mb_artist** na camada Bronze apenas com os campos que vou precisar utilizar no estudo.

In [0]:
%skip
%sql
USE CATALOG mvp;
USE SCHEMA bronze;

###Canonical Dump Music Brainz

Salvando a tabela com os dados da **canonical_musicbranz_data.csv** como **musicbrainz**:

In [0]:
%skip
df_musicbrainz = spark.read.csv(
    "/Volumes/mvp/staging/imdb/canonical_musicbrainz_data.csv",
    header=True,
    inferSchema=True
)

In [0]:
%skip
df_musicbrainz.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("mvp.bronze.musicbrainz")

In [0]:
%sql
SELECT *
FROM mvp.bronze.musicbrainz
LIMIT 10

O **Canonical Dump** continha ainda duas outras data frames: **canonical_recording_redirect.csv** e **canonical_release_redirect.csv**. Esses dados não serão necessários ao estudo e as dataframes **não foram salvas em tabelas na camada Bronze**.

###Spotify Huge Audio Features

Salvando a tabela com os dados de **spotify-huge-audio-features.parquet** como **spotify**:

In [0]:
%skip
df_spotify = spark.read.parquet(
    "/Volumes/mvp/staging/imdb/spotify-huge-audio-features.parquet"
)

In [0]:
%skip
df_spotify.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("mvp.bronze.spotify")

In [0]:
%sql
SELECT *
FROM mvp.bronze.spotify
LIMIT 10

### Music Brainz Area
Escolhi não usar as informações de **area** no projeto, ela **não foi salva em tabela na camada bronze**.

### Music Brainz Artist

A JSON **artist** de Music Brainz atende ao propósito de informar um país relacionado a um artista, contendo os **mbids** dos artitas, campo que será usado como **chave** para unir a informação de localização às demais informações de **musicbrainz**.

Os dados foram salvos na tabela **mb_artist**:

In [0]:
df_mb_artist = spark.read.json(
    "/Volumes/mvp/staging/imdb/artist"
)

Essa base de dados contém nomes com caracteres não permitidos. Estou investigando como salvar fazendo o mínimo de modificações:

In [0]:
df_mb_artist.printSchema()

In [0]:
import re

for coluna in df_mb_artist.columns:
    encontrados = re.findall(r'[ ,;{}()\n\t=]', coluna)
    if encontrados:
        print(repr(coluna), encontrados)

In [0]:
display(df_mb_artist.limit(1))

O problema está nos campos internos do JSON, e não nas colunas principais. Dentro de relations.attribute-credits e relations.attribute-ids, existem campos com espaços, parênteses, barras, apóstrofos etc.

Considerando a complexidade desnecessária dessa transformação já que desses dados só serão usadas as colunas **id**, **name** e **country**, eu optei por **salvar apenas as colunas que serão usadas no estudo, sem fazer nenhuma transformação**.

O dado bruto continua íntegro na camada **Staging**, mas muitas colunas dessa dataframe não foram importadas como tabela na camada **Bronze**.

In [0]:
df_mb_artist_bronze = df_mb_artist.select(
    "id",
    "name",
    "country"
)

In [0]:
df_mb_artist_bronze.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("mvp.bronze.mb_artist")

In [0]:
%sql
SELECT *
FROM mvp.bronze.mb_artist
LIMIT 10

### Comentando a Tabela musicbrainz

In [0]:
%sql
COMMENT ON TABLE mvp.bronze.musicbrainz
IS 'Dados canonical do MusicBrainz contendo informações de artistas, lançamentos e gravações, usados como base para cruzamento com os dados do Spotify'

In [0]:
table = "mvp.bronze.musicbrainz"
columns = [
    ("id", "Identificador único do registro"),
    ("artist_credit_id", "Identificador do crédito do artista no MusicBrainz"),
    ("artist_mbids", "MBIDs dos artistas associados"),
    ("artist_credit_name", "Nome do crédito do artista"),
    ("release_mbid", "MBID do lançamento (release)"),
    ("release_name", "Nome do lançamento (release)"),
    ("recording_mbid", "MBID da gravação (recording)"),
    ("recording_name", "Nome da gravação (recording)"),
    ("combined_lookup", "Chave de busca combinada para cruzamento de dados"),
    ("score", "Pontuação de similaridade do cruzamento de dados")
]

for col, comment in columns:
    spark.sql(f"COMMENT ON COLUMN {table}.{col} IS '{comment}'")

In [0]:
%sql
DESCRIBE mvp.bronze.musicbrainz;

### Comentando a Tabela spotify

In [0]:
%sql
COMMENT ON TABLE mvp.bronze.spotify
IS 'Dados de faixas do Spotify com características de áudio (danceability, energy, tempo, etc.), metadados de álbum e artista, e métricas de popularidade'

In [0]:
table = "mvp.bronze.spotify"
columns = [
    ("track_id", "Identificador único da faixa no Spotify"),
    ("artist_name", "Nome do artista"),
    ("track_name", "Nome da faixa"),
    ("album_name", "Nome do álbum"),
    ("album_release_date", "Data de lançamento do álbum"),
    ("duration_ms", "Duração da faixa em milissegundos"),
    ("explicit", "Indica se a faixa contém conteúdo explícito (1 = sim, 0 = não)"),
    ("track_number", "Número da faixa no álbum"),
    ("disc_number", "Número do disco no álbum"),
    ("track_popularity", "Popularidade da faixa no Spotify (0-100)"),
    ("album_popularity", "Popularidade do álbum no Spotify (0-100)"),
    ("track_vs_album_popularity", "Razão entre a popularidade da faixa e a do álbum"),
    ("artist_popularity", "Popularidade do artista no Spotify (0-100)"),
    ("artist_followers", "Número de seguidores do artista no Spotify"),
    ("album_vs_artist_popularity", "Razão entre a popularidade do álbum e a do artista"),
    ("tempo", "Tempo da faixa em BPM (batidas por minuto)"),
    ("key", "Tonalidade da faixa usando notação de pitch class"),
    ("mode", "Modalidade da faixa (0 = menor, 1 = maior)"),
    ("danceability", "Quão dançável a faixa é (0.0-1.0)"),
    ("energy", "Energia perceptível da faixa (0.0-1.0)"),
    ("loudness", "Volume geral da faixa em decibéis (dB)"),
    ("speechiness", "Presença de palavras faladas na faixa (0.0-1.0)"),
    ("acousticness", "Quão acústica a faixa é (0.0-1.0)"),
    ("instrumentalness", "Quão instrumental a faixa é (0.0-1.0)"),
    ("liveness", "Probabilidade de a faixa ter sido gravada ao vivo (0.0-1.0)"),
    ("valence", "Positividade musical da faixa (0.0-1.0)"),
    ("energy_danceability_score", "Pontuação combinada de energia e dançabilidade")
]

for col, comment in columns:
    spark.sql(f"COMMENT ON COLUMN {table}.{col} IS '{comment}'")

In [0]:
%sql
DESCRIBE mvp.bronze.spotify;

### Comentando a Tabela mb_artist

In [0]:
%sql
COMMENT ON TABLE mvp.bronze.mb_artist
IS 'Dados de artistas do MusicBrainz com MBID, nome e país relacionado ao artista, usados para enriquecer a tabela musicbrainz com a localização'

In [0]:
table = "mvp.bronze.mb_artist"
columns = [
    ("id", "Identificador único do artista no MusicBrainz"),
    ("name", "Nome do artista"),
    ("country", "Principal país associado ao artista")
]

for col, comment in columns:
    spark.sql(f"COMMENT ON COLUMN {table}.{col} IS '{comment}'")

In [0]:
%sql
DESCRIBE mvp.bronze.mb_artist;